# Recreate 4-point and 6-point EC fits

This notebook walks through a directory containing one text file per sample, extracts paired pore-water conductivity (`EC_water`) and bulk conductivity (`EC_bulk`), and fits

\[
EC_{bulk} = \frac{1}{F} EC_{water} + \sigma_s,
\]

where `F = 1 / slope` is the formation factor and `σs = intercept` is the surface conductivity.

It compares two explicitly configurable fits:

- **4-point fit**: by default the first four measurements in each file, matching the four concentrations in the final report as closely as the raw files allow.
- **6-point fit**: by default all six measurements.

The notebook also optionally reads the previously generated comparison workbook, joins the reported final-report and paper values, calculates residual differences, creates one plot per sample, and writes a combined CSV.

**Important:** The example file already shows that “first four” versus “all six” does not reproduce either published value exactly. The notebook therefore treats the point-selection rule as a testable configuration rather than assuming it is the full explanation.

In [49]:

from pathlib import Path
import re
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# ---- USER SETTINGS ----
REVIL_DIR = Path(r"c:\Gitrepos\formationfactors\data\1-external\revil")
DATA_DIR = REVIL_DIR / "data"          # directory containing sample .txt files
FILE_GLOB = "*.txt"
REFERENCE_XLSX =REVIL_DIR / "Revil_results_consistency.xlsx"  # optional
OUTPUT_DIR = REVIL_DIR / "fit_check_output"
OUTPUT_DIR.mkdir(exist_ok=True)

# Point-selection rules. Alternatives: "lowest", "highest", or explicit targets.
FOUR_POINT_RULE = "highest"
N_FOUR = 4
N_SIX = 3

# Optional target EC_water values. Set to None to use sorted point counts.
FOUR_TARGETS = None  # e.g. [0.031, 0.526, 1.037, 4.17]
SIX_TARGETS = [5.7, 14.7, 22.0]
TARGET_RTOL = 0.25   # maximum relative deviation for nearest-target matching

PLOT_EACH_SAMPLE = True
SHOW_PLOTS = True


In [50]:

def read_sample_file(path: Path) -> pd.DataFrame:
    """Read the first two numeric columns from a tab/space/comma/semicolon-delimited file."""
    rows = []
    for line in path.read_text(errors="replace").splitlines():
        tokens = [t for t in re.split(r"[\t,; ]+", line.strip()) if t]
        nums = []
        for t in tokens:
            try:
                nums.append(float(t.replace(",", ".")))
            except ValueError:
                pass
        if len(nums) >= 2:
            rows.append(nums[:2])
    if not rows:
        raise ValueError(f"No paired numeric columns found in {path.name}")
    df = pd.DataFrame(rows, columns=["EC_water", "EC_bulk"])
    return df.dropna().sort_values("EC_water").reset_index(drop=True)


def select_points(df, n=None, rule="lowest", targets=None, rtol=0.25):
    if targets is not None:
        chosen = []
        available = df.copy()
        for target in targets:
            if available.empty:
                break
            i = (available.EC_water - target).abs().idxmin()
            row = available.loc[i]
            rel = abs(row.EC_water - target) / max(abs(target), 1e-12)
            if rel > rtol:
                raise ValueError(f"No EC_water close enough to target {target}; nearest={row.EC_water}")
            chosen.append(row)
            available = available.drop(i)
        return pd.DataFrame(chosen).reset_index(drop=True)
    if n is None or len(df) < n:
        return df.copy()
    return (df.nsmallest(n, "EC_water") if rule == "lowest"
            else df.nlargest(n, "EC_water").sort_values("EC_water"))


def fit_linear(df):
    """OLS fit EC_bulk = slope * EC_water + intercept."""
    x = df.EC_water.to_numpy(float)
    y = df.EC_bulk.to_numpy(float)
    slope, intercept = np.polyfit(x, y, 1)
    pred = slope * x + intercept
    ss_res = np.sum((y - pred)**2)
    ss_tot = np.sum((y - y.mean())**2)
    r2 = np.nan if ss_tot == 0 else 1 - ss_res / ss_tot
    return {"n": len(df), "slope": slope, "F": 1/slope, "SC": intercept,
            "R2": r2, "RMSE": np.sqrt(np.mean((y-pred)**2)),
            "EC_min": x.min(), "EC_max": x.max()}


In [51]:

# Discover and inspect input files
files = sorted(DATA_DIR.glob(FILE_GLOB))
print(f"Found {len(files)} input files in {DATA_DIR.resolve()}")
if files:
    display(read_sample_file(files[0]))


Found 71 input files in C:\Gitrepos\formationfactors\data\1-external\revil\data


,EC_water,EC_bulk
0,0.03142,0.28105
1,0.52610,0.57666
2,1.03700,0.33541
3,4.16500,1.54540
4,9.78000,3.07550
5,14.04000,4.58210


In [52]:

# Optional reference values from the comparison workbook
reference = None
if REFERENCE_XLSX.exists():
    reference = pd.read_excel(REFERENCE_XLSX, sheet_name="Comparison")
    reference = reference.rename(columns={
        "Code Revil": "sample",
        "F final report": "F_reported_final",
        "SC final report": "SC_reported_final",
        "F paper": "F_reported_paper",
        "SC paper": "SC_reported_paper",
    })
    reference = reference[["sample", "F_reported_final", "SC_reported_final",
                           "F_reported_paper", "SC_reported_paper"]]
    display(reference.head())
else:
    print("Reference workbook not found; raw fits will still be produced.")


,sample,F_reported_final,SC_reported_final,F_reported_paper,SC_reported_paper
0,CW_0580586130,5.32,0.247,8.37,0.30
1,AE_0580586137,4.65,0.248,7.83,0.33
2,CT_0580586129,2.21,0.223,3.97,0.45
3,CS_0580586138,5.81,0.354,8.30,0.37
4,CV_0580586132,7.08,0.311,10.40,0.28


In [53]:

results = []
errors = []
for path in files:
    sample = path.stem
    try:
        raw = read_sample_file(path)
        p4 = select_points(raw, n=N_FOUR, rule=FOUR_POINT_RULE,
                           targets=FOUR_TARGETS, rtol=TARGET_RTOL)
        p6 = select_points(raw, n=N_SIX, rule="lowest",
                           targets=SIX_TARGETS, rtol=TARGET_RTOL)
        f4, f6 = fit_linear(p4), fit_linear(p6)
        row = {"sample": sample, "file": path.name, "n_raw": len(raw)}
        row.update({f"{k}_4pt": v for k, v in f4.items()})
        row.update({f"{k}_6pt": v for k, v in f6.items()})
        results.append(row)

        if PLOT_EACH_SAMPLE:
            fig, ax = plt.subplots(figsize=(7.2, 5.0))
            ax.scatter(raw.EC_water, raw.EC_bulk, c="0.65", s=45, label="All observations", zorder=2)
            ax.scatter(p4.EC_water, p4.EC_bulk, marker="o", s=70, facecolors="none",
                       edgecolors="#0072B2", linewidths=1.7, label="4-point selection", zorder=3)
            ax.scatter(p6.EC_water, p6.EC_bulk, marker="x", s=55, color="#D55E00",
                       label="6-point selection", zorder=4)
            xx = np.linspace(0, raw.EC_water.max()*1.04, 200)
            ax.plot(xx, f4["slope"]*xx + f4["SC"], color="#0072B2",
                    label=f'4-point: F={f4["F"]:.3g}, SC={f4["SC"]:.3g}, R²={f4["R2"]:.3f}')
            ax.plot(xx, f6["slope"]*xx + f6["SC"], color="#D55E00",
                    label=f'6-point: F={f6["F"]:.3g}, SC={f6["SC"]:.3g}, R²={f6["R2"]:.3f}')
            ax.set(xlabel="EC_water (S/m)", ylabel="EC_bulk (S/m)", title=sample)
            ax.grid(alpha=0.25); ax.legend(fontsize=8)
            fig.tight_layout(); fig.savefig(OUTPUT_DIR / f"{sample}_fits.png", dpi=180)
            if SHOW_PLOTS: plt.show()
            plt.close(fig)
    except Exception as exc:
        errors.append({"sample": sample, "file": path.name, "error": str(exc)})

results = pd.DataFrame(results)
if reference is not None and not results.empty:
    results = results.merge(reference, on="sample", how="left")
    for fit, reported in [("4pt", "final"), ("6pt", "paper")]:
        results[f"dF_{fit}_vs_{reported}"] = results[f"F_{fit}"] - results[f"F_reported_{reported}"]
        results[f"dSC_{fit}_vs_{reported}"] = results[f"SC_{fit}"] - results[f"SC_reported_{reported}"]
        results[f"abs_rel_F_{fit}_vs_{reported}"] = abs(results[f"dF_{fit}_vs_{reported}"] / results[f"F_reported_{reported}"])
        results[f"abs_rel_SC_{fit}_vs_{reported}"] = abs(results[f"dSC_{fit}_vs_{reported}"] / results[f"SC_reported_{reported}"].replace(0, np.nan))

results.to_csv(OUTPUT_DIR / "fit_comparison.csv", index=False)
pd.DataFrame(errors).to_csv(OUTPUT_DIR / "read_errors.csv", index=False)
print(f"Processed {len(results)} samples; {len(errors)} errors")
display(results.head())


Processed 0 samples; 71 errors


""


In [54]:

# Overall diagnostic: does 4-point → final and 6-point → paper hold?
if reference is not None and not results.empty:
    cols = ["sample", "F_4pt", "F_reported_final", "SC_4pt", "SC_reported_final",
            "F_6pt", "F_reported_paper", "SC_6pt", "SC_reported_paper",
            "dF_4pt_vs_final", "dSC_4pt_vs_final", "dF_6pt_vs_paper", "dSC_6pt_vs_paper"]
    display(results[cols].sort_values("sample"))

    summary = pd.DataFrame({
        "comparison": ["4-point vs final report", "6-point vs paper"],
        "median_abs_relative_F": [results.abs_rel_F_4pt_vs_final.median(), results.abs_rel_F_6pt_vs_paper.median()],
        "median_abs_relative_SC": [results.abs_rel_SC_4pt_vs_final.median(), results.abs_rel_SC_6pt_vs_paper.median()],
        "max_abs_F_difference": [results.dF_4pt_vs_final.abs().max(), results.dF_6pt_vs_paper.abs().max()],
        "max_abs_SC_difference": [results.dSC_4pt_vs_final.abs().max(), results.dSC_6pt_vs_paper.abs().max()],
    })
    display(summary)
    summary.to_csv(OUTPUT_DIR / "fit_summary.csv", index=False)


## Interpretation checks

1. If the recreated 4-point values closely equal the final-report columns and the recreated 6-point values closely equal the paper columns, the hypothesis is supported.
2. If the 4-point and 6-point fits move in the right direction but do not reproduce the reported values, point count contributes but is not the complete explanation. Check the exact concentration subset, temperature corrections, equilibration measurements, frequency used for `EC_bulk`, and whether the published fit excluded the nonlinear low-salinity point.
3. If the match improves when `FOUR_POINT_RULE` is changed to `"highest"`, the final fit likely used the linear high-salinity branch rather than the first four chronological measurements.
4. Use `FOUR_TARGETS` and `SIX_TARGETS` to reproduce named brine concentrations rather than relying on row order.